In [ ]:
from google.colab import files
uploaded = files.upload()
!pip install --quiet optuna



Saving sleep_mobile_stress_dataset_15000.csv to sleep_mobile_stress_dataset_15000 (2).csv


In [ ]:
import pandas as pd
import numpy as np
import torch

df=pd.read_csv('sleep_mobile_stress_dataset_15000.csv')

In [ ]:
df.describe()

,user_id,age,daily_screen_time_hours,phone_usage_before_sleep_minutes,sleep_duration_hours,sleep_quality_score,stress_level,caffeine_intake_cups,physical_activity_minutes,notifications_received_per_day,mental_fatigue_score
count,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.00000,15000.000000,15000.000000,15000.000000
mean,7500.500000,38.488467,5.501528,59.708933,6.509683,6.246362,6.980247,1.99880,59.157133,160.890467,6.873009
std,4330.271354,12.007970,2.600085,34.641858,1.452689,1.713644,2.749382,1.41459,34.525705,80.856902,2.730482
min,1.000000,18.000000,1.000000,0.000000,4.000000,1.000000,1.000000,0.00000,0.000000,20.000000,1.000000
25%,3750.750000,28.000000,3.260000,29.000000,5.260000,5.000000,4.750000,1.00000,29.000000,92.000000,4.700000
50%,7500.500000,38.000000,5.490000,60.000000,6.490000,6.250000,7.380000,2.00000,59.000000,162.000000,7.380000
75%,11250.250000,49.000000,7.760000,90.000000,7.790000,7.500000,10.000000,3.00000,89.000000,231.000000,9.450000
max,15000.000000,59.000000,10.000000,119.000000,9.000000,10.000000,10.000000,4.00000,119.000000,299.000000,10.000000


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 13 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   user_id                           15000 non-null  int64  
 1   age                               15000 non-null  int64  
 2   gender                            15000 non-null  object 
 3   occupation                        15000 non-null  object 
 4   daily_screen_time_hours           15000 non-null  float64
 5   phone_usage_before_sleep_minutes  15000 non-null  int64  
 6   sleep_duration_hours              15000 non-null  float64
 7   sleep_quality_score               15000 non-null  float64
 8   stress_level                      15000 non-null  float64
 9   caffeine_intake_cups              15000 non-null  int64  
 10  physical_activity_minutes         15000 non-null  int64  
 11  notifications_received_per_day    15000 non-null  int64  
 12  ment

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder


categorical_columns = ['occupation']
encoder = OneHotEncoder(sparse_output=False)
one_hot_encoded = encoder.fit_transform(df[categorical_columns])
one_hot_df = pd.DataFrame(one_hot_encoded, columns=encoder.get_feature_names_out(categorical_columns))
data = pd.concat([df, one_hot_df], axis=1)
data = data.drop(['occupation'], axis=1)


label=pd.DataFrame()
label['gender']= df['gender']
le = LabelEncoder()
label['gender'] = le.fit_transform(data['gender'])
data['gender']=label['gender']

In [ ]:
#Keeping things simple lets pretend 5 or above is stressed
stressed=pd.DataFrame()
stressed['stressed_binary']= (data['stress_level'] >= 5).astype(int)
data['stress_level']=stressed['stressed_binary']
data.head()
data=data.drop('user_id', axis=1)

In [ ]:
from sklearn.model_selection import train_test_split

y=data.loc[:,'stress_level']
data=data.drop('stress_level',axis=1)
X=data.iloc[:,:]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

In [ ]:
if torch.cuda.is_available():
  print("Using CUDA (GPU)")
  device = torch.device("cuda")
elif torch.backends.mps.is_available():
  print("Using MPS (macOS)")
  device = xm.xla_device()
else:
  print("Using CPU (Default)")
  device = torch.device("cpu")

Using CUDA (GPU)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

columns_to_scale = [
    'age',
    'daily_screen_time_hours',
    'phone_usage_before_sleep_minutes',
    'sleep_duration_hours',
    'sleep_quality_score',
    'caffeine_intake_cups',
    'physical_activity_minutes',
    'notifications_received_per_day',
    'mental_fatigue_score'
]

# Apply StandardScaler to the selected columns in X_train
X_train[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])

scaler = StandardScaler()
X_test[columns_to_scale] = scaler.fit_transform(X_test[columns_to_scale])


print("X_train head after scaling:")
print(X_train.head())
print(X_train.shape)


X_train head after scaling:
            age  gender  daily_screen_time_hours  \
655    0.891502       1                 0.801027   
12044 -0.612669       1                -1.440122   
14844  0.640807       0                 0.142545   
13985  0.306547       2                 1.590435   
7974   1.309328       0                -0.720028   

       phone_usage_before_sleep_minutes  sleep_duration_hours  \
655                           -1.115685              1.651104   
12044                         -1.173476             -1.464337   
14844                         -0.682252             -0.375311   
13985                          1.484916             -0.899146   
7974                          -0.162131              1.092806   

       sleep_quality_score  caffeine_intake_cups  physical_activity_minutes  \
655               0.914397             -0.706547                   0.891557   
12044             0.506570             -0.706547                  -0.816384   
14844             0.401700     

In [ ]:
import torch
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import optuna


X_train_tensor=torch.Tensor(X_train.to_numpy())
y_train_tensor = torch.LongTensor(y_train.to_numpy())
X_test_tensor = torch.Tensor(X_test.to_numpy())
y_test_tensor = torch.LongTensor(y_test.to_numpy())

class SimpleModel(nn.Module):
    def __init__(self, trial):
        super(SimpleModel, self).__init__()
        self.flatten = nn.Flatten()

        in_features = 18

        n_layers = trial.suggest_int("n_layers", 1, 3)

        layers = []
        for i in range(n_layers):
            out_features = trial.suggest_int(f"n_units_l{i}", 4, 128)
            layers.append(nn.Linear(in_features, out_features))
            layers.append(nn.ReLU())
            in_features = out_features
        layers.append(nn.Linear(in_features, 2))

        self.classifier = nn.Sequential(*layers) # Changed from self.output = nn.Linear(*layers)

    def forward(self, x):
        x = self.flatten(x)
        x = self.classifier(x)
        return x


train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
def objective(trial):
  # Instantiate the model for the current trial
  model = SimpleModel(trial)

  criterion = nn.CrossEntropyLoss()

  # Optimize learning rate
  lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
  optimizer = optim.Adam(model.parameters(), lr=lr)

  model.to(device) # Move model to GPU or MPS if available

  # Training Loop
  num_epochs = 10
  for epoch in range(num_epochs):
    model.train() # Set model to training mode
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    for inputs, labels in train_loader:
      inputs, labels = inputs.to(device), labels.to(device) # Move to GPU if available

      # Forward pass
      outputs = model(inputs)
      loss = criterion(outputs, labels)

      # Backward and optimize
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      train_loss += loss.item()
      _, predicted = torch.max(outputs.data, 1)
      total_train += labels.size(0)
      correct_train += (predicted == labels).sum().item()

    print(f'Epoch {epoch+1}/{num_epochs}, Train Loss:{train_loss/len(train_loader):.4f}, Train Acc: {100*correct_train/total_train:.4f}%')

  # Return the accuracy for Optuna to maximize
  return correct_train / total_train

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)
trial = study.best_trial

print("Accuracy: {}".format(trial.value))
print("Best hyperparameters: {}".format(trial.params))

[I 2026-04-30 13:58:46,053] A new study created in memory with name: no-name-c6d4ca5b-a281-44b8-bd67-1c63bdb6842d


Epoch 1/10, Train Loss:0.1614, Train Acc: 93.3632%
Epoch 2/10, Train Loss:0.1360, Train Acc: 93.9502%
Epoch 3/10, Train Loss:0.1380, Train Acc: 94.1294%
Epoch 4/10, Train Loss:0.1341, Train Acc: 94.1493%
Epoch 5/10, Train Loss:0.1340, Train Acc: 94.2090%
Epoch 6/10, Train Loss:0.1328, Train Acc: 94.0896%
Epoch 7/10, Train Loss:0.1297, Train Acc: 94.2886%
Epoch 8/10, Train Loss:0.1261, Train Acc: 94.2488%
Epoch 9/10, Train Loss:0.1314, Train Acc: 93.8308%


[I 2026-04-30 13:58:49,720] Trial 0 finished with value: 0.9418905472636816 and parameters: {'n_layers': 3, 'n_units_l0': 12, 'n_units_l1': 102, 'n_units_l2': 29, 'lr': 0.04355367059450542}. Best is trial 0 with value: 0.9418905472636816.


Epoch 10/10, Train Loss:0.1334, Train Acc: 94.1891%
Epoch 1/10, Train Loss:0.6680, Train Acc: 69.4229%
Epoch 2/10, Train Loss:0.5828, Train Acc: 83.5821%
Epoch 3/10, Train Loss:0.4924, Train Acc: 86.8657%
Epoch 4/10, Train Loss:0.4062, Train Acc: 90.7861%
Epoch 5/10, Train Loss:0.3334, Train Acc: 92.7960%
Epoch 6/10, Train Loss:0.2772, Train Acc: 93.4129%
Epoch 7/10, Train Loss:0.2371, Train Acc: 93.9303%
Epoch 8/10, Train Loss:0.2084, Train Acc: 94.0398%
Epoch 9/10, Train Loss:0.1866, Train Acc: 94.0896%


[I 2026-04-30 13:58:53,056] Trial 1 finished with value: 0.9407960199004975 and parameters: {'n_layers': 2, 'n_units_l0': 119, 'n_units_l1': 61, 'lr': 2.798733693510214e-05}. Best is trial 0 with value: 0.9418905472636816.


...

Epoch 10/10, Train Loss:0.1164, Train Acc: 94.7861%
Epoch 1/10, Train Loss:0.1634, Train Acc: 92.7463%
Epoch 2/10, Train Loss:0.1408, Train Acc: 94.0498%
Epoch 3/10, Train Loss:0.1321, Train Acc: 94.1393%
Epoch 4/10, Train Loss:0.1300, Train Acc: 94.3980%
Epoch 5/10, Train Loss:0.1361, Train Acc: 94.0100%
Epoch 6/10, Train Loss:0.1344, Train Acc: 94.3085%
Epoch 7/10, Train Loss:0.1301, Train Acc: 94.3085%
Epoch 8/10, Train Loss:0.1236, Train Acc: 94.4378%
Epoch 9/10, Train Loss:0.1354, Train Acc: 94.5075%


[I 2026-04-30 13:59:16,827] Trial 8 finished with value: 0.9378109452736318 and parameters: {'n_layers': 3, 'n_units_l0': 51, 'n_units_l1': 29, 'n_units_l2': 75, 'lr': 0.033520128892969694}. Best is trial 3 with value: 0.9499502487562189.


Epoch 10/10, Train Loss:0.1409, Train Acc: 93.7811%
Epoch 1/10, Train Loss:0.4337, Train Acc: 81.6517%
Epoch 2/10, Train Loss:0.1498, Train Acc: 93.8607%
Epoch 3/10, Train Loss:0.1307, Train Acc: 94.1493%
Epoch 4/10, Train Loss:0.1271, Train Acc: 94.2886%
Epoch 5/10, Train Loss:0.1252, Train Acc: 94.3980%
Epoch 6/10, Train Loss:0.1238, Train Acc: 94.4776%
Epoch 7/10, Train Loss:0.1226, Train Acc: 94.5473%
Epoch 8/10, Train Loss:0.1213, Train Acc: 94.6667%
Epoch 9/10, Train Loss:0.1210, Train Acc: 94.5970%


[I 2026-04-30 13:59:20,766] Trial 9 finished with value: 0.9471641791044776 and parameters: {'n_layers': 3, 'n_units_l0': 54, 'n_units_l1': 111, 'n_units_l2': 70, 'lr': 0.00019579229993843563}. Best is trial 3 with value: 0.9499502487562189.


Epoch 10/10, Train Loss:0.1199, Train Acc: 94.7164%
Accuracy: 0.9499502487562189
Best hyperparameters: {'n_layers': 2, 'n_units_l0': 107, 'n_units_l1': 25, 'lr': 0.011717945745445265}


In [ ]:
optuna.visualization.plot_optimization_history(study)

In [ ]:
optuna.visualization.plot_slice(study)

ValueError: Parameter n_estimators does not exist in your study.